In [1]:
from dotenv import load_dotenv
load_dotenv('env')

True

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, GenerationConfig, AutoModel, TextStreamer
from peft import PeftModel, AutoPeftModelForCausalLM
# from unsloth import FastLanguageModel
import torch
from datasets import load_dataset

In [3]:
BASE_MODEL_ID = 'deepseek-ai/deepseek-math-7b-base'
LORA_ADAPTER_ID = 'Hrushi/AIMO-SFT-DDP-deepseek-math-7b-base-N141128-E1-R128-A256'
SAVE_FPATH = "AIMO_Finetuned_Models/AIMO-SFT-DDP-Deepseek-Math-7B-base-N141128-E1-R128-A256"

In [4]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype='float16',
)


In [6]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    use_cache=True,
    # device_map={"": 0},
    device_map="auto",
    quantization_config=bnb_config,
)

config.json:   0%|          | 0.00/594 [00:00<?, ?B/s]

pytorch_model.bin.index.json:   0%|          | 0.00/22.5k [00:00<?, ?B/s]

pytorch_model-00001-of-00002.bin:   0%|          | 0.00/9.97G [00:00<?, ?B/s]

pytorch_model-00002-of-00002.bin:   0%|          | 0.00/3.85G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

In [7]:
base_model.dequantize()

The model is going to be dequantized in torch.float16 - if you want to upcast it to another dtype, make sure to pass the desired dtype when quantizing the model through `bnb_4bit_quant_type` argument of `BitsAndBytesConfig`
For some reason the model has not been properly dequantized. You might see unexpected behavior.


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(102400, 4096)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )
    )
    (norm): LlamaRMSNorm()
  )
  (lm_head)

In [8]:
peft_model = PeftModel.from_pretrained(base_model, LORA_ADAPTER_ID, device_map="auto")

# peft_model = AutoPeftModelForCausalLM.from_pretrained(
#     LORA_ADAPTER_ID,
#     torch_dtype=torch.bfloat16,
#     # use_cache=True,
#     device_map={"": 0},
#     quantization_config=bnb_config,
#     low_cpu_mem_usage=True,
#     attn_implementation='flash_attention_2'
# )

# unsloth_model, _ = FastLanguageModel.from_pretrained(
#     model_name=LORA_ADAPTER_ID,
#     # model_name=SAVE_FPATH,
    
# )

adapter_config.json:   0%|          | 0.00/738 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

In [9]:
tokenizer = AutoTokenizer.from_pretrained(LORA_ADAPTER_ID)

tokenizer_config.json:   0%|          | 0.00/824 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.61M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/369 [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [12]:
tokenizer.chat_template = "{{ bos_token }}{% set step_counter = namespace(value=1) %}{% for message in messages %}{% if (message['role'] == 'system') %}{{ message['content'] | trim + '\n\n' }}{% elif (message['role'] == 'problem') %}{{ '<math_problem>' + message['content'] | trim + '</math_problem>\n' }}{% elif (message['role'] == 'final_answer') %}{{ '<end_of_solution>\n<final_answer>' + message['content'] | trim + '</final_answer>' }}{% else %}{% if step_counter.value == 1 %}{{ '<start_of_solution>\n' }}{% endif %}{{ '<start_of_step>' + step_counter.value | string + '\n' }}{% for sub_message in message['content'] %}{% if (sub_message['role'] == 'text') %}{{ sub_message['content'] | trim }}{% elif (sub_message['role'] == 'code') %}{{ '<code_block>' + sub_message['content'] | trim + '</code_block>\n' }}{% elif (sub_message['role'] == 'code_output') %}{{ '<code_output_block>'  + sub_message['content'] | trim + '</code_output_block>\n' }}{% endif %}{% endfor %}{{ '<end_of_step>\n' }}{% set step_counter.value = step_counter.value + 1 %}{% endif %}{% endfor %}{% if add_generation_prompt %}{{ '<start_of_solution>\n' }}{% endif %}"

In [10]:
tokenizer.tokenize('<start_of_solution>')

['<', 'start', '_', 'of', '_', 'solution', '>']

In [9]:
system_prompt = """Your are a high school student appearing for your math exam.
You will be given a math problem to solve, and your job is to write the detailed step by step solution, with good mathematical reasoning and using python code for calculations, simplifications, solving equations, etc.

Format help and instructions:
- Problem has been given enclosed in <math_problem></math_problem> tags.
- Every step must be written within <start_of_step><end_of_step> tags.
- Code needs to be always written inside <code_block></code_block> tags - No backticks or triple backticks.
- Code output will be given in <code_output_block></code_output_block> tags

Every calculation should be performed using writing supporting python code."""

In [8]:
MATH_ds = load_dataset('hendrycks/competition_math', trust_remote_code=True)['test']
MATH_df = MATH_ds.to_pandas()

In [17]:
generation_config = GenerationConfig.from_pretrained(BASE_MODEL_ID)
generation_config.update(**{
    'max_length': 4096,
    'max_new_tokens': 4096,
    'stop_strings': ['<code_output_block>', '</final_answer>'],
    'do_sample': True,
    'temperature': 0.2,
    'top_p': 1.0,
    'top_k': 100,
    'num_return_sequences': 1,
    'use_cache': True,
})

{}

In [18]:
%%time
idx = 0
problem = MATH_df['problem'][idx]

messages = [
    {'role': 'system', 'content': system_prompt},
    {'role': 'problem', 'content': problem}
]


input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True).replace(tokenizer.bos_token, '')
print(input_text)
input_tokens = tokenizer(input_text, return_tensors='pt').to('cuda')
output_tokens = peft_model.generate(**input_tokens, generation_config=generation_config, tokenizer=tokenizer, streamer=TextStreamer(tokenizer))

# raw_output = tokenizer.batch_decode(output_tokens)[0]
# raw_output = raw_output.replace('<bos>', '').replace(input_text, '').replace('<pad>', '')
# print(raw_output)

Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.


Your are a high school student appearing for your math exam.
You will be given a math problem to solve, and your job is to write the detailed step by step solution, with good mathematical reasoning and using python code for calculations, simplifications, solving equations, etc.

Format help and instructions:
- Problem has been given enclosed in <math_problem></math_problem> tags.
- Every step must be written within <start_of_step><end_of_step> tags.
- Code needs to be always written inside <code_block></code_block> tags - No backticks or triple backticks.
- Code output will be given in <code_output_block></code_output_block> tags

Every calculation should be performed using writing supporting python code.

<math_problem>How many vertical asymptotes does the graph of $y=\frac{2}{x^2+x-6}$ have?</math_problem>
<start_of_solution>

<｜begin▁of▁sentence｜>Your are a high school student appearing for your math exam.
You will be given a math problem to solve, and your job is to write the detai

In [19]:
merged_model = peft_model.merge_and_unload(progressbar=True, safe_merge=True)

Unloading and merging model: 100%|██████████| 636/636 [00:00<00:00, 708.55it/s]


In [20]:
%%time
output_tokens = merged_model.generate(**input_tokens, generation_config=generation_config, tokenizer=tokenizer, streamer=TextStreamer(tokenizer))

Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.


<｜begin▁of▁sentence｜>Your are a high school student appearing for your math exam.
You will be given a math problem to solve, and your job is to write the detailed step by step solution, with good mathematical reasoning and using python code for calculations, simplifications, solving equations, etc.

Format help and instructions:
- Problem has been given enclosed in <math_problem></math_problem> tags.
- Every step must be written within <start_of_step><end_of_step> tags.
- Code needs to be always written inside <code_block></code_block> tags - No backticks or triple backticks.
- Code output will be given in <code_output_block></code_output_block> tags

Every calculation should be performed using writing supporting python code.

<math_problem>How many vertical asymptotes does the graph of $y=\frac{2}{x^2+x-6}$ have?</math_problem>
<start_of_solution>
<start_of_step>1
The graph of $y=\frac{2}{x^2+x-6}$ has vertical asymptotes where the denominator is equal to zero.
So, we need to find the

In [19]:
# !rm -rf $SAVE_FPATH

In [21]:
merged_model.save_pretrained(save_directory=SAVE_FPATH)
tokenizer.save_pretrained(save_directory=SAVE_FPATH)

('AIMO_Finetuned_Models/AIMO-SFT-DDP-Deepseek-Math-7B-base-N141128-E1-R128-A256/tokenizer_config.json',
 'AIMO_Finetuned_Models/AIMO-SFT-DDP-Deepseek-Math-7B-base-N141128-E1-R128-A256/special_tokens_map.json',
 'AIMO_Finetuned_Models/AIMO-SFT-DDP-Deepseek-Math-7B-base-N141128-E1-R128-A256/tokenizer.json')

In [4]:
import json
import os

In [5]:
with open(os.path.join(SAVE_FPATH, 'config.json'), 'r') as f:
    data = json.load(f)
data

{'_name_or_path': 'deepseek-ai/deepseek-math-7b-base',
 'architectures': ['LlamaForCausalLM'],
 'attention_bias': False,
 'attention_dropout': 0.0,
 'bos_token_id': 100000,
 'eos_token_id': 100001,
 'hidden_act': 'silu',
 'hidden_size': 4096,
 'initializer_range': 0.02,
 'intermediate_size': 11008,
 'max_position_embeddings': 4096,
 'mlp_bias': False,
 'model_type': 'llama',
 'num_attention_heads': 32,
 'num_hidden_layers': 30,
 'num_key_value_heads': 32,
 'pretraining_tp': 1,
 'rms_norm_eps': 1e-06,
 'rope_scaling': None,
 'rope_theta': 10000.0,
 'tie_word_embeddings': False,
 'torch_dtype': 'float16',
 'transformers_version': '4.41.2',
 'use_cache': True,
 'vocab_size': 102400}

In [24]:
data.pop('quantization_config')

{'_load_in_4bit': True,
 '_load_in_8bit': False,
 'bnb_4bit_compute_dtype': 'float16',
 'bnb_4bit_quant_storage': 'uint8',
 'bnb_4bit_quant_type': 'nf4',
 'bnb_4bit_use_double_quant': True,
 'llm_int8_enable_fp32_cpu_offload': False,
 'llm_int8_has_fp16_weight': False,
 'llm_int8_skip_modules': None,
 'llm_int8_threshold': 6.0,
 'load_in_4bit': True,
 'load_in_8bit': False,
 'quant_method': 'bitsandbytes'}

In [26]:
with open(os.path.join(SAVE_FPATH, 'config.json'), 'w') as f:
    json.dump(data, f, indent=4)

In [4]:
model = AutoModelForCausalLM.from_pretrained(SAVE_FPATH, device_map="cuda", torch_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(SAVE_FPATH)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [5]:
generation_config = GenerationConfig.from_pretrained(BASE_MODEL_ID)
generation_config.update(**{
    'max_length': 4096,
    'max_new_tokens': 4096,
    'stop_strings': ['<code_output_block>', '</final_answer>'],
    'do_sample': True,
    'temperature': 0.2,
    'top_p': 1.0,
    'top_k': 100,
    'num_return_sequences': 1,
    'use_cache': True,
})

{}

In [6]:
SAVE_FPATH.split('/')[1]

'AIMO-SFT-DDP-Deepseek-Math-7B-base-N141128-E1-R128-A256'

In [10]:
idx = 0
problem = MATH_df['problem'][idx]

messages = [
    {'role': 'system', 'content': system_prompt},
    {'role': 'problem', 'content': problem}
]


input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True).replace(tokenizer.bos_token, '')
print(input_text)
input_tokens = tokenizer(input_text, return_tensors='pt').to('cuda')
model.generate(**input_tokens, generation_config=generation_config, streamer=TextStreamer(tokenizer), tokenizer=tokenizer)

Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.


Your are a high school student appearing for your math exam.
You will be given a math problem to solve, and your job is to write the detailed step by step solution, with good mathematical reasoning and using python code for calculations, simplifications, solving equations, etc.

Format help and instructions:
- Problem has been given enclosed in <math_problem></math_problem> tags.
- Every step must be written within <start_of_step><end_of_step> tags.
- Code needs to be always written inside <code_block></code_block> tags - No backticks or triple backticks.
- Code output will be given in <code_output_block></code_output_block> tags

Every calculation should be performed using writing supporting python code.

<math_problem>How many vertical asymptotes does the graph of $y=\frac{2}{x^2+x-6}$ have?</math_problem>
<start_of_solution>

<｜begin▁of▁sentence｜>Your are a high school student appearing for your math exam.
You will be given a math problem to solve, and your job is to write the detai

tensor([[100000,   7616,    418,    245,   1461,   2696,   6710,  20606,    327,
            520,  16145,   3799,     13,    185,   2054,    543,    330,   2028,
            245,  16145,   2066,    276,   8708,     11,    285,    520,   3112,
            317,    276,   3708,    254,   9333,   3458,    457,   3458,   3418,
             11,    366,   1207,  23668,  22834,    285,   1244,   9934,   2985,
            327,  14365,     11,   7667,   8445,     11,  17693,   8935,     11,
           3387,     13,    185,    185,   8609,   1345,    285,   9454,     25,
            185,     12,  23363,    643,    803,   2028,  33024,    279,    459,
            664,     62,  25114,   1575,    664,     62,  25114,     29,  15983,
             13,    185,     12,   6576,   3458,   1534,    330,   4473,   2383,
            459,   4789,     62,    994,     62,   9215,   1611,    409,     62,
            994,     62,   9215,     29,  15983,     13,    185,     12,  10578,
           3277,    276,    

In [11]:
model.push_to_hub(repo_id="Hrushi/AIMO-SFT-DDP-Merged-Deepseek-Math-7B-Base-N141128-E1-R128-A256", private=True)
tokenizer.push_to_hub(repo_id="Hrushi/AIMO-SFT-DDP-Merged-Deepseek-Math-7B-Base-N141128-E1-R128-A256", private=True)

model-00003-of-00003.safetensors:   0%|          | 0.00/3.85G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Upload 3 LFS files:   0%|          | 0/3 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Hrushi/AIMO-SFT-DDP-Merged-Deepseek-Math-7B-Base-N141128-E1-R128-A256/commit/28e19e8a1c9e1defaaa8a82c0e683cead1d658bc', commit_message='Upload tokenizer', commit_description='', oid='28e19e8a1c9e1defaaa8a82c0e683cead1d658bc', pr_url=None, pr_revision=None, pr_num=None)

In [1]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from datasets import load_dataset
from dotenv import load_dotenv
import torch
load_dotenv('env')

True

In [2]:
ds = load_dataset('Hrushi/SFT-MetaMath-MATH-TrainingData')['train']
df = ds.to_pandas()

In [10]:
tokenizer = AutoTokenizer.from_pretrained('Hrushi/AIMO-SFT-DDP-Merged-Deepseek-Math-7B-Base-N141128-E1-R128-A256')
sampling_params = SamplingParams(
    n=2,
    temperature=0.2,
    top_p=1,
    top_k=100,
    max_tokens=4096,
    skip_special_tokens=False,
    spaces_between_special_tokens=False,
    stop=['<code_output_block>', '<final_answer>']
)

tokenizer.tokenize('<start_of_solution>')

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


['<', 'start', '_', 'of', '_', 'solution', '>']

In [4]:
%%time
llm = LLM(
    model='Hrushi/AIMO-SFT-DDP-Merged-Deepseek-Math-7B-Base-N141128-E1-R128-A256',
    tokenizer='Hrushi/AIMO-SFT-DDP-Merged-Deepseek-Math-7B-Base-N141128-E1-R128-A256',
    kv_cache_dtype='fp8',
    gpu_memory_utilization=0.95,
    tensor_parallel_size=torch.cuda.device_count(),
    max_model_len=4096,
    trust_remote_code=True,
    # adapter_name_or_path=LORA_ADAPTER_ID
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/opt/conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/769 [00:00<?, ?B/s]

INFO 06-10 06:19:03 config.py:390] Using fp8 data type to store kv cache. It reduces the GPU memory footprint and boosts the performance. Meanwhile, it may cause accuracy drop without a proper scaling factor
INFO 06-10 06:19:03 llm_engine.py:161] Initializing an LLM engine (v0.4.3) with config: model='Hrushi/AIMO-SFT-DDP-Merged-Deepseek-Math-7B-Base-N141128-E1-R128-A256', speculative_config=None, tokenizer='Hrushi/AIMO-SFT-DDP-Merged-Deepseek-Math-7B-Base-N141128-E1-R128-A256', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, rope_scaling=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=fp8, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), seed=0, served_model_name=Hrushi/AIMO-SFT-DDP-Merged-Deepseek-Ma

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/3.85G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/22.5k [00:00<?, ?B/s]

INFO 06-10 06:20:35 model_runner.py:146] Loading model weights took 12.8725 GB
INFO 06-10 06:20:36 gpu_executor.py:83] # GPU blocks: 6355, # CPU blocks: 1092
INFO 06-10 06:20:40 model_runner.py:854] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 06-10 06:20:40 model_runner.py:858] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 06-10 06:20:49 model_runner.py:924] Graph capturing finished in 9 secs.
CPU times: user 34.6 s, sys: 29.3 s, total: 1min 3s
Wall time: 1min 45s


In [5]:
prompts = [df.prompt[0]]
prompts

['Your are a high school student appearing for your math exam.\nYou will be given a math problem to solve, and your job is to write the detailed step by step solution, with good mathematical reasoning and using python code for calculations, simplifications, solving equations, etc.\n\nFormat help and instructions:\n- Problem has been given enclosed in <math_problem></math_problem> tags.\n- Every step must be written within <start_of_step><end_of_step> tags.\n- Code needs to be always written inside <code_block></code_block> tags - No backticks or triple backticks.\n- Code output will be given in <code_output_block></code_output_block> tags.\n\n<math_problem>What is the total cost of purchasing equipment for all sixteen players on the football team, considering that each player requires a $25 jersey, a $15.20 pair of shorts, and a pair of socks priced at $6.80?</math_problem>\n<start_of_solution>\n']

In [12]:
response = llm.generate(prompts,sampling_params=sampling_params )

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.80s/it, Generation Speed: 117.06 toks/s]


In [13]:
print(response[0].outputs[0].text)

<start_of_step>1
Each player requires a $25 jersey, a $15.20 pair of shorts, and a pair of socks priced at $6.80.
So, the total cost for each player is $25 + $15.20 + $6.80 =$<code_block>print(25 + 15.20 + 6.80)</code_block>



In [15]:
print(25 + 15.20 + 6.80)

47.0


In [14]:
print(df.completion[0])

<start_of_step>1
Each player requires a $25 jersey, a $15.20 pair of shorts, and a pair of socks priced at $6.80.
So the total cost for each player is $25 + $15.20 + $6.80 =$<code_block>print(25 + 15.20 + 6.80)</code_block>
<code_output_block>47.0</code_output_block>
Hence, the total cost for each player is $\boxed{47}$ dollars.<end_of_step>
<start_of_step>2
Since there are sixteen players on the football team, the total cost for all of them is $16 \times $47 =$<code_block>print(16 * 47)</code_block>
<code_output_block>752</code_output_block>
Hence, the total cost for all sixteen players is $\boxed{752}$ dollars.<end_of_step>
<end_of_solution>
<final_answer>752</final_answer>
